# 13 · Maintenance loss & interaction features

**Decision: diagnose the rejected WATER-deferral policy; do not rerun or promote it.**

Notebook 12 lost 396 own coins. This notebook reconstructs the same recorded actions and evaluates crop-interaction features separately from outcome ledgers. No new policy decisions, model fits, or Kaggle submission occur. The first views use verified prior results; later views require a real AWS replay.

In [1]:
from pathlib import Path
import sys, json
BASE = Path.home() / 'kaggriculture_maintenance_diagnosis'
assert (BASE / 'run_diagnosis.py').is_file(), 'Open the extracted package in your existing AWS space.'
sys.path.insert(0, str(BASE))
from artifact_io import read, sha
import pandas as pd
from IPython.display import display
print('PACKAGE:', BASE)

PACKAGE: /home/sagemaker-user/kaggriculture_maintenance_diagnosis


## 1. The result we are preserving

A completed negative result is not a crash. Neither the earlier +cash intervals nor fewer WATER commands justify promotion. These tables are from the uploaded notebook-12 evidence.

In [2]:
review = read(BASE / 'reference/input_review.json')
print('Prior bundle hashes verified:', review['bundle_hashes_verified'])
prior = read(BASE / 'reference/notebook12/outputs/pilot/report.json')
display(pd.DataFrame(prior['paired_results']))

Prior bundle hashes verified: 29


,arm,coin_margin_candidate,coin_margin_control,coin_margin_delta,coins_candidate,coins_control,coins_delta,local_match_score_candidate,local_match_score_control,local_match_score_delta,...,opponent_coins_control,opponent_coins_delta,residual_product_units_candidate,residual_product_units_control,residual_product_units_delta,same_state_action_changes,seat,seed,water_commands_candidate,water_commands_control
0,coordinated,603.0,758.0,-155.0,44444.0,44840.0,-396.0,1.0,1.0,0.0,...,44082.0,-241.0,0,0,0,15,0,1601,68,78


In [3]:
from visualize import figures
prior_figures = figures(BASE)[:2]
for fig in prior_figures:
    fig.show()

## 2. Prospectively defined features

The new representation describes WATER–HARVEST interactions, two-refresh survival buffers and current worker access. Four local action scenarios do not incorporate travel or solve joint routing. Availability masks prevent hypothetical post-terminal benefits. The accounting ledger uses evaluator-only state and never enters these feature inputs.

In [4]:
dictionary = pd.read_csv(BASE / 'feature_dictionary.csv')
display(dictionary[['feature','level','status']])
print('Plant descriptors:', (dictionary.level == 'plant').sum(), '| State summaries:', (dictionary.level == 'state').sum())

,feature,level,status
0,scenario_available,plant,candidate; no policy use or predictive-value c...
1,pass_units,plant,candidate; no policy use or predictive-value c...
2,water_units,plant,candidate; no policy use or predictive-value c...
3,harvest_units,plant,candidate; no policy use or predictive-value c...
4,water_harvest_units,plant,candidate; no policy use or predictive-value c...
5,water_gain_without_harvest,plant,candidate; no policy use or predictive-value c...
6,water_gain_with_harvest,plant,candidate; no policy use or predictive-value c...
7,water_harvest_interaction,plant,candidate; no policy use or predictive-value c...
8,cap_masked_water_gain,plant,candidate; no policy use or predictive-value c...
9,two_refresh_available,plant,candidate; no policy use or predictive-value c...


Plant descriptors: 22 | State summaries: 8


## 3. Run the single bounded diagnostic

The worker verifies the prior reports, raw checkpoint and pinned source, compares 480 artificial crop scenarios with installed engine components, then replays each recorded branch. Every cash trace and both original final-state hashes must match. **120-second cap**, ten-second heartbeats, per-branch checkpoints. No failed-stage automatic retry. If this cell fails, save the notebook and run the bundle command in START_HERE.md.

In [5]:
import subprocess
console = BASE / 'outputs/notebook_console.txt'
console.parent.mkdir(exist_ok=True)
with console.open('a') as log:
    process = subprocess.Popen([sys.executable, str(BASE / 'run_diagnosis.py'), 'run'], cwd=BASE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        raise
if returncode:
    raise RuntimeError('Diagnosis stopped. Save and bundle; do not retry unchanged.')

..............................................
----------------------------------------------------------------------
Ran 46 tests in 0.992s

OK
{"utc": "2026-09-12T03:02:39.297046+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10, "recorded_actions": 478}
{"utc": "2026-09-12T03:02:39.396670+00:00", "stage": "MECHANICS_PASSED", "component_scenarios": 480}
{"utc": "2026-09-12T03:02:39.935636+00:00", "stage": "PREFIX_REPLAY", "mode": "control", "completed": 120, "total": 480}
{"utc": "2026-09-12T03:02:40.371662+00:00", "stage": "PREFIX_REPLAY", "mode": "control", "completed": 240, "total": 480}
{"utc": "2026-09-12T03:02:40.803302+00:00", "stage": "PREFIX_REPLAY", "mode": "control", "completed": 360, "total": 480}
{"utc": "2026-09-12T03:02:41.337847+00:00", "stage": "PREFIX_REPLAY", "mode": "control", "completed": 480, "total": 480}
{"utc": "2026-09-12T03:02:41.849289+00:00", "stage": "RECORDED_SUFFIX_REPLAY", "mode": "control", "completed": 48, "total": 239}
{"utc": "2026-09-12

## 4. Acceptance and cash reconciliation

Success means reproducing the old −396-coin endpoint and explaining its cash components. It does not mean an improved agent. A price/quantity decomposition is descriptive because prices responded to each trajectory.

In [6]:
from run_diagnosis import verify_completed
report = verify_completed()
for key in ('status','decision','cash_control','cash_defer','cash_delta_reconciled','all_turn_cash_and_final_state_checks_passed','new_policy_calls','new_interventions','official_submission_score'):
    print(f'{key}: {report[key]}')
assert report['cash_delta_reconciled'] == -396.0
assert report['new_policy_calls'] == 0

status: MAINTENANCE_DIAGNOSIS_COMPLETE
decision: DIAGNOSIS_ONLY_NO_POLICY_PROMOTION
cash_control: 44840.0
cash_defer: 44444.0
cash_delta_reconciled: -396.0
all_turn_cash_and_final_state_checks_passed: True
new_policy_calls: 0
new_interventions: 0
official_submission_score: None


In [7]:
ledger = pd.read_csv(BASE / 'outputs/cash_decomposition.csv')
display(ledger[ledger.player == 0].sort_values('cash_delta'))
print('Own-cash ledger sum:', ledger.loc[ledger.player == 0, 'cash_delta'].sum())

,player,operation,item,cash_control,cash_defer,units_control,units_defer,cash_delta,units_delta,quantity_component,average_price_component
5,0,SELL,MILK,3996.0,3531.0,28,28,-465.0,0,0.000000,-465.000000
8,0,SELL,WHEAT,2766.0,2669.0,59,57,-97.0,-2,-93.705917,-3.294083
7,0,SELL,TOMATO,1923.0,1861.0,24,24,-62.0,0,0.000000,-62.000000
1,0,BUY_SEED,WHEAT,-60.0,-70.0,6,7,-10.0,1,NaN,NaN
6,0,SELL,STRAWBERRY,1046.0,1044.0,4,4,-2.0,0,0.000000,-2.000000
2,0,HIRE,LABOR,-40.0,-40.0,30,30,0.0,0,NaN,NaN
9,0,SELL,WOOL,6866.0,6867.0,28,28,1.0,0,0.000000,1.000000
0,0,BUY_PRODUCT,WHEAT,-2185.0,-2181.0,48,48,4.0,0,NaN,NaN
3,0,SELL,EGG,1893.0,1915.0,44,44,22.0,0,0.000000,22.000000
4,0,SELL,FERTILIZER,2608.0,2821.0,45,49,213.0,4,231.053968,-18.053968


Own-cash ledger sum: -396.0


## 5. Crop mechanism and prospective feature evidence

A missing crop or a change in average sale price is not inferred from the cash curve alone. Use the reconstructed committed trades and crop events. Feature activation is not importance and is not evidence for an outcome gain.

In [8]:
crop_events = pd.read_csv(BASE / 'outputs/crop_event_summary.csv')
display(crop_events[crop_events.player == 0])
registry = pd.read_csv(BASE / 'outputs/feature_registry.csv')
display(registry.sort_values('nonzero_fraction', ascending=False))

,mode,player,crop,event,units
0,control,0,STRAWBERRY,fertilize,0.0
1,control,0,STRAWBERRY,harvest,4.0
2,control,0,STRAWBERRY,night_production,4.0
3,control,0,STRAWBERRY,water,0.0
4,control,0,TOMATO,decay_death,0.0
5,control,0,TOMATO,fertilize,0.0
6,control,0,TOMATO,harvest,24.0
7,control,0,TOMATO,night_production,24.0
8,control,0,TOMATO,water,0.0
9,control,0,WHEAT,fertilize,0.0


,feature,distinct_values,nonzero_fraction,status
16,single_worker_water_actions,6,1.000000,descriptive_activation_not_predictive_validation
13,current_water_deadline_callbacks,64,1.000000,descriptive_activation_not_predictive_validation
17,current_water_deadline_slack,69,0.998912,descriptive_activation_not_predictive_validation
19,current_yield_headroom,6,0.998369,descriptive_activation_not_predictive_validation
15,second_worker_travel,7,0.983143,descriptive_activation_not_predictive_validation
21,next_refresh_available,2,0.946982,descriptive_activation_not_predictive_validation
18,workers_with_water_access,5,0.930941,descriptive_activation_not_predictive_validation
0,scenario_available,2,0.907287,descriptive_activation_not_predictive_validation
14,nearest_worker_travel,6,0.872213,descriptive_activation_not_predictive_validation
9,two_refresh_available,2,0.870854,descriptive_activation_not_predictive_validation


In [9]:
from visualize import save_dashboard
dashboard, all_figures = save_dashboard(BASE)
for fig in all_figures[2:]:
    fig.show()
print('Saved standalone Plotly dashboard:', dashboard)

Saved standalone Plotly dashboard: /home/sagemaker-user/kaggriculture_maintenance_diagnosis/outputs/maintenance_diagnosis_dashboard.html


## 6. Interpretation and next decision

Do not adopt a policy from these diagnostics. A crop-interaction hypothesis must explain the measured loss channel and then pass its own controlled action and endpoint ablation. All current rows are exploratory development evidence from one selected source; no untouched seed or opponent generalization is claimed. Save with **Ctrl+S**, create the results ZIP in the terminal, and stop the AWS application without deleting its space.

In [10]:
summary = {'status':'NOTEBOOK13_COMPLETE','figures':len(all_figures),'prior_feature_intervention':'REJECTED','new_feature_value_proven':False,'new_policy_calls':report['new_policy_calls'],'report_sha256':sha(BASE/'outputs/report.json'),'dashboard_sha256':sha(dashboard)}
from artifact_io import write
write(BASE/'outputs/notebook_review.json', summary)
print(json.dumps(summary, indent=2))

{
  "status": "NOTEBOOK13_COMPLETE",
  "figures": 8,
  "prior_feature_intervention": "REJECTED",
  "new_feature_value_proven": false,
  "new_policy_calls": 0,
  "report_sha256": "b37699a64e26d3740564324e2740b2737bbc72e4a30dc2686c8e2e25a5444a59",
  "dashboard_sha256": "ba69f776efae2c6005786f15b6fdaf94d304c7e871246d00e2589b6ab0733f4b"
}
